In [17]:
from pathlib import Path
import numpy as np
import pandas as pd

# Índices COCO-17
L_SHOULDER, R_SHOULDER = 5, 6
L_HIP, R_HIP = 11, 12
L_KNEE, R_KNEE = 13, 14
L_ANKLE, R_ANKLE = 15, 16

TARGET_FPS = 25.0

df_index = pd.read_csv("data/processed/pose_index.csv")

print(df_index.groupby(["dataset", "clase_video"])["fps"].describe())

fps_npz = []
for p in df_index["pose_path"]:
    with np.load(p) as data:
        fps_npz.append(float(data["fps"]))

df_index["fps_npz"] = fps_npz
print("Diferencia máxima CSV vs NPZ:", (df_index["fps"] - df_index["fps_npz"]).abs().max())

                     count       mean       std     min     25%        50%  \
dataset clase_video                                                          
gavd    normal        19.0  27.875789  2.831152  24.000  24.500  29.969999   
        parkinsons    22.0  26.977000  3.071631  23.976  23.976  26.973000   
urfall  adl           25.0  30.000000  0.000000  30.000  30.000  30.000000   
        fall          10.0  30.000000  0.000000  30.000  30.000  30.000000   

                           75%        max  
dataset clase_video                        
gavd    normal       29.969999  29.969999  
        parkinsons   29.974001  30.000000  
urfall  adl          30.000000  30.000000  
        fall         30.000000  30.000000  
Diferencia máxima CSV vs NPZ: 0.0


In [18]:
def remuestrear_secuencia(kpts_xy: np.ndarray, fps_orig: float, fps_target: float = 25.0) -> np.ndarray:
    """Remuestrea (T, 17, 2) a fps_target manteniendo los huecos NaN."""
    if np.isclose(fps_orig, fps_target, atol=0.5):
        return kpts_xy.astype(np.float32)

    T, J, C = kpts_xy.shape
    duracion = (T - 1) / fps_orig
    T_new = int(np.floor(duracion * fps_target)) + 1
    t_orig = np.linspace(0, duracion, T)
    t_new = np.linspace(0, duracion, T_new)

    out = np.full((T_new, J, C), np.nan, dtype=np.float32)
    for j in range(J):
        for c in range(C):
            y = kpts_xy[:, j, c]
            valid = ~np.isnan(y)
            if valid.sum() >= 2:
                y_res = np.interp(t_new, t_orig[valid], y[valid])
                mask_nan = np.interp(t_new, t_orig, np.isnan(y).astype(float)) > 0.5
                y_res[mask_nan] = np.nan
                out[:, j, c] = y_res
    return out

In [19]:
def interpolar_cortos(kpts_xy: np.ndarray, max_gap: int = 8) -> np.ndarray:
    """Rellena huecos de NaN cortos (<= max_gap frames, ~0.32s a 25Hz)."""
    out = kpts_xy.copy()
    T, J, C = out.shape
    t = np.arange(T)

    for j in range(J):
        for c in range(C):
            y = out[:, j, c]
            valid = ~np.isnan(y)
            if valid.sum() < 2:
                continue
            isnan = ~valid
            y_interp = np.interp(t, t[valid], y[valid])
            idx_change = np.where(np.diff(isnan.astype(int)) != 0)[0] + 1
            for seg in np.split(np.arange(T), idx_change):
                if isnan[seg[0]] and len(seg) <= max_gap:
                    out[seg, j, c] = y_interp[seg]
    return out


def normalizar_esqueleto(kpts_xy: np.ndarray) -> np.ndarray:
    """Centra en la cadera media y escala por la longitud del torso."""
    mid_hip = 0.5 * (kpts_xy[:, L_HIP, :] + kpts_xy[:, R_HIP, :])          # (T, 2)
    mid_shoulder = 0.5 * (kpts_xy[:, L_SHOULDER, :] + kpts_xy[:, R_SHOULDER, :])

    torso_len = np.linalg.norm(mid_shoulder - mid_hip, axis=1)             # (T,)
    escala_fallback = np.nanmedian(torso_len)
    if np.isnan(escala_fallback) or escala_fallback < 1e-3:
        escala_fallback = 1.0

    torso_len = np.where(np.isnan(torso_len) | (torso_len < 1e-3), escala_fallback, torso_len)
    kpts_norm = (kpts_xy - mid_hip[:, None, :]) / torso_len[:, None, None]
    return kpts_norm.astype(np.float32)

In [20]:
def calcular_freeze_index_y_vel(
    win_kpts: np.ndarray,
    fps: float = 25.0,
    eps: float = 1e-6
) -> dict:
    """
    Calcula métricas del baseline sobre una ventana (W, 17, 2) ya normalizada.
    Usa la norma 2D del desplazamiento de tobillos para resistir cambios de cam_view.
    """
    # Velocidad instantánea de ambos tobillos (en torsos/segundo)
    ankles = win_kpts[:, [L_ANKLE, R_ANKLE], :]          # (W, 2, 2)
    vel_2d = np.linalg.norm(np.diff(ankles, axis=0), axis=-1) * fps  # (W-1, 2)
    vel_media = float(np.mean(vel_2d))

    # Señal de velocidad vertical (y) promedio de ambos tobillos para el espectro
    vel_y = np.mean(np.diff(ankles[:, :, 1], axis=0), axis=1) * fps  # (W-1,)
    vel_y = vel_y - np.mean(vel_y)  # Eliminar componente DC

    N = len(vel_y)
    ventana_hann = np.hanning(N)
    psd = np.abs(np.fft.rfft(vel_y * ventana_hann)) ** 2
    freqs = np.fft.rfftfreq(N, d=1.0 / fps)

    mask_loco = (freqs >= 0.5) & (freqs < 3.0)
    mask_freeze = (freqs >= 3.0) & (freqs <= 8.0)

    pot_loco = float(np.sum(psd[mask_loco]))
    pot_freeze = float(np.sum(psd[mask_freeze]))
    fi = pot_freeze / (pot_loco + eps)

    return {
        "vel_tobillo_media": vel_media,
        "pot_loco_05_3hz": pot_loco,
        "pot_freeze_3_8hz": pot_freeze,
        "freeze_index": fi,
    }


def extraer_ventanas_clip(
    npz_path: str,
    win_sec: float = 2.5,
    step_sec: float = 0.5,
    fps_target: float = 25.0,
    min_valid_ratio: float = 0.85,
) -> pd.DataFrame:
    with np.load(npz_path) as data:
        kpts_xy = data["kpts_xy"]
        fps_orig = float(data["fps"])

    kpts_res = remuestrear_secuencia(kpts_xy, fps_orig, fps_target)
    kpts_imp = interpolar_cortos(kpts_res, max_gap=int(0.32 * fps_target))
    kpts_norm = normalizar_esqueleto(kpts_imp)

    W = int(round(win_sec * fps_target))
    S = int(round(step_sec * fps_target))
    T = len(kpts_norm)

    filas = []
    for start in range(0, T - W + 1, S):
        end = start + W
        win = kpts_norm[start:end]

        # Filtrar ventanas donde falten tobillos o caderas en >15% de frames
        req_joints = win[:, [L_HIP, R_HIP, L_ANKLE, R_ANKLE], :]
        valid_frames = ~np.isnan(req_joints).any(axis=(1, 2))
        if valid_frames.mean() < min_valid_ratio:
            continue

        # Si queda algún NaN residual (<15%), rellenar por borde dentro de la ventana
        if not valid_frames.all():
            win = pd.DataFrame(win.reshape(W, -1)).ffill().bfill().to_numpy().reshape(W, 17, 2)

        feats = calcular_freeze_index_y_vel(win, fps=fps_target)
        feats["start_frame"] = start
        feats["end_frame"] = end
        filas.append(feats)

    return pd.DataFrame(filas)

In [21]:
ventanas_list = []

for _, row in df_index.iterrows():
    df_w = extraer_ventanas_clip(
        npz_path=row["pose_path"],
        win_sec=2.5,
        step_sec=0.5,
        fps_target=TARGET_FPS,
        min_valid_ratio=0.85,
    )
    if df_w.empty:
        continue

    df_w["clip_id"] = row["clip_id"] if "clip_id" in df_index.columns else Path(row["pose_path"]).stem
    df_w["dataset"] = row["dataset"]
    df_w["clase_video"] = row["clase_video"]
    ventanas_list.append(df_w)

df_ventanas = pd.concat(ventanas_list, ignore_index=True)

print(f"Total de ventanas válidas extraídas: {len(df_ventanas)}")
print(df_ventanas.groupby(["dataset", "clase_video"])["clip_id"].agg(clips="nunique", ventanas="count"))

Total de ventanas válidas extraídas: 774
                     clips  ventanas
dataset clase_video                 
gavd    normal          13       144
        parkinsons      21       359
urfall  adl             24       223
        fall            10        48


In [22]:
# 1. Estadísticos descriptivos por grupo
resumen_baseline = df_ventanas.groupby(["dataset", "clase_video"])[
    ["vel_tobillo_media", "freeze_index"]
].agg(["mean", "median", lambda x: np.percentile(x, 90)]).rename(columns={"<lambda_0>": "p90"})
print(resumen_baseline)

# 2. Calibrar umbrales de referencia usando marcha normal (gavd/normal)
normal_df = df_ventanas[(df_ventanas["dataset"] == "gavd") & (df_ventanas["clase_video"] == "normal")]

# Bloqueo con temblor: Freeze Index por encima del percentil 95 de marcha normal
UMBRAL_FI = normal_df["freeze_index"].quantile(0.95)
# Bloqueo acinético: velocidad de tobillos muy baja (< percentil 5 de marcha normal)
UMBRAL_VEL_BAJA = normal_df["vel_tobillo_media"].quantile(0.05)

df_ventanas["pred_fog_temblor"] = df_ventanas["freeze_index"] > UMBRAL_FI
df_ventanas["pred_fog_acinetico"] = df_ventanas["vel_tobillo_media"] < UMBRAL_VEL_BAJA
df_ventanas["pred_fog_baseline"] = df_ventanas["pred_fog_temblor"] | df_ventanas["pred_fog_acinetico"]

print(f"\nUmbrales calibrados -> FI > {UMBRAL_FI:.3f} | Vel < {UMBRAL_VEL_BAJA:.3f} torsos/s")
print("\nTasa de ventanas detectadas como bloqueo por el baseline (%):")
print(
    df_ventanas.groupby(["dataset", "clase_video"])[
        ["pred_fog_temblor", "pred_fog_acinetico", "pred_fog_baseline"]
    ].mean().mul(100).round(2)
)

# 3. Guardar resultados del baseline
out_csv = Path("data/processed/ventanas_baseline.csv")
df_ventanas.to_csv(out_csv, index=False)
print(f"\nResultados guardados en: {out_csv}")

                    vel_tobillo_media                      freeze_index  \
                                 mean    median        p90         mean   
dataset clase_video                                                       
gavd    normal               0.699732  0.602558   1.336990     2.808515   
        parkinsons           0.525434  0.463110   0.827442     2.623987   
urfall  adl                  1.776523  1.298374   3.253954     3.578281   
        fall                 7.126398  1.999029  24.055279     1.894119   

                                         
                       median       p90  
dataset clase_video                      
gavd    normal       2.090098  5.051249  
        parkinsons   1.976800  5.326595  
urfall  adl          1.970841  8.262982  
        fall         1.117255  4.211356  

Umbrales calibrados -> FI > 5.661 | Vel < 0.145 torsos/s

Tasa de ventanas detectadas como bloqueo por el baseline (%):
                     pred_fog_temblor  pred_fog_acinetico  